# Cycle 3 — Tuning: Player Injury Risk

In [1]:
import sys, os               
import pandas as pd          
import numpy as np           
import warnings; import joblib 
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

_here = os.getcwd()                                     
while not os.path.isdir(os.path.join(_here, 'data')):   
    _p = os.path.dirname(_here)                         
    if _p == _here: raise RuntimeError('project root not found')  
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)                             

from config import Paths, ensure_dirs
ensure_dirs()  


In [2]:
df = pd.read_csv(str(Paths.PLAYER_INJURIES_PROCESSED))  # load preprocessed injury dataset
X = df.drop(columns=['High_Injury'])   # 17 pre-season features
y = df['High_Injury']                  # target: 0=low, 1=high injury burden

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # same split as modelling notebook
)

scaler = StandardScaler()                      # z-score scaler
X_train_s = scaler.fit_transform(X_train)       # fit on train — no leakage
X_test_s  = scaler.transform(X_test)

spw = y_train.value_counts()[0] / y_train.value_counts()[1]  # class weight ratio for XGB/LGB

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # inner CV for all searches

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'scale_pos_weight: {spw:.2f}')
print(f'Features: {len(X.columns)}')


Train: 1040 | Test: 261
scale_pos_weight: 0.42
Features: 17


## XGBoost Hyperparameter Tuning

In [8]:
xgb_param_grid = {
    'n_estimators':     [100, 200, 300],               # boosting rounds
    'max_depth':        [3, 4, 5, 6],                  # tree depth
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],        # shrinkage
    'subsample':        [0.6, 0.7, 0.8, 1.0],          # row subsampling
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],          # feature subsampling
    'min_child_weight': [1, 3, 5],                     # min samples per leaf
    'gamma':            [0, 0.1, 0.2],                 # min loss reduction for split
    'scale_pos_weight': [spw, 0.5, 0.637, 1.0],        # class weight variants to try
}

xgb_base = XGBClassifier(random_state=42, eval_metric='auc', verbosity=0)
xgb_search = RandomizedSearchCV(
    xgb_base, xgb_param_grid,
    n_iter=50, scoring='roc_auc',    # 50 combinations, scored by AUC
    cv=cv, random_state=42, n_jobs=-1
)
xgb_search.fit(X_train_s, y_train)  # run the search

xgb_best = xgb_search.best_estimator_             # best model found
y_prob_xgb_tuned = xgb_best.predict_proba(X_test_s)[:,1]  # injury probabilities
y_pred_xgb_tuned = xgb_best.predict(X_test_s)    # class labels

print('XGBOOST TUNED')
print(f'  Best CV AUC:  {xgb_search.best_score_:.4f}')   # CV AUC
print(f'  Test AUC:     {roc_auc_score(y_test, y_prob_xgb_tuned):.4f}')  # held-out AUC
print(f'  Test Acc:     {accuracy_score(y_test, y_pred_xgb_tuned)*100:.2f}%')
print(f'  Best params:  {xgb_search.best_params_}')
print(classification_report(y_test, y_pred_xgb_tuned, target_names=['Low Injury','High Injury']))


XGBOOST TUNED
  Best CV AUC:  0.6584
  Test AUC:     0.6558
  Test Acc:     63.22%
  Best params:  {'subsample': 1.0, 'scale_pos_weight': np.float64(0.4246575342465753), 'n_estimators': 100, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.2, 'gamma': 0, 'colsample_bytree': 0.8}
              precision    recall  f1-score   support

  Low Injury       0.40      0.49      0.44        78
 High Injury       0.76      0.69      0.73       183

    accuracy                           0.63       261
   macro avg       0.58      0.59      0.58       261
weighted avg       0.65      0.63      0.64       261



- CV AUC=0.6584, Test AUC=0.6558 — the gap is only (0.003), meaning the tuned model generalises well to held-out data
- XGBoost tuned (0.6558) beats the Logistic Regression baseline (0.6220) by +0.034 AUC — a meaningful improvement
- Best params favour a shallow tree (max_depth=4), high learning_rate=0.2, few estimators (100) — consistent with a small dataset that cannot support deep or complex models
- min_child_weight=5 and subsample=1.0 act as light regularisation to prevent overfitting

## Random Forest Hyperparameter Tuning

In [4]:
rf_param_grid = {
    'n_estimators':      [100, 200, 300],         # number of trees
    'max_depth':         [5, 10, 15, None],        # None = fully grown
    'min_samples_split': [2, 5, 10],              # min samples to split a node
    'min_samples_leaf':  [1, 2, 4],               # min samples per leaf
    'max_features':      ['sqrt', 'log2'],          # features per split
    'class_weight':      ['balanced', 'balanced_subsample'],  # imbalance handling
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_search = RandomizedSearchCV(
    rf_base, rf_param_grid,
    n_iter=50, scoring='roc_auc',
    cv=cv, random_state=42, n_jobs=-1
)
rf_search.fit(X_train_s, y_train)  # run the search

rf_best = rf_search.best_estimator_
y_prob_rf_tuned = rf_best.predict_proba(X_test_s)[:,1]
y_pred_rf_tuned = rf_best.predict(X_test_s)

print('RANDOM FOREST TUNED')
print(f'  Best CV AUC:  {rf_search.best_score_:.4f}')
print(f'  Test AUC:     {roc_auc_score(y_test, y_prob_rf_tuned):.4f}')
print(f'  Test Acc:     {accuracy_score(y_test, y_pred_rf_tuned)*100:.2f}%')
print(f'  Best params:  {rf_search.best_params_}')
print(classification_report(y_test, y_pred_rf_tuned, target_names=['Low Injury','High Injury']))


RANDOM FOREST TUNED
  Best CV AUC:  0.6411
  Test AUC:     0.6169
  Test Acc:     69.35%
  Best params:  {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 10, 'class_weight': 'balanced'}
              precision    recall  f1-score   support

  Low Injury       0.48      0.31      0.38        78
 High Injury       0.74      0.86      0.80       183

    accuracy                           0.69       261
   macro avg       0.61      0.58      0.59       261
weighted avg       0.67      0.69      0.67       261



- CV AUC=0.6411, Test AUC=0.6169 — CV-to-test gap is 0.024, a real overfitting-to-CV signal (unlike XGBoost whose gap was only 0.003)
- Substantial improvement over untuned RF (0.5916 → 0.6169) — the baseline RF had majority-class collapse; min_samples_leaf=4 
- Tuned RF is now a real discriminator, but still below both Logistic Regression (0.6220) and XGBoost tuned (0.6558)
- Random Forest benefits less from tuning on this small dataset — its ensemble of uncorrelated trees needs more data to outperform boosting

## LightGBM Hyperparameter Tuning

Runs `RandomizedSearchCV` with 50 combinations over an 8-parameter LightGBM grid, scored by AUC-ROC via 5-fold stratified CV.
- LightGBM had the highest baseline AUC (0.6355). Tuning it may produce the best injury-risk model overall.


In [5]:
lgb_param_grid = {
    'n_estimators':     [100, 200, 300],               # boosting rounds
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],        # shrinkage
    'max_depth':        [3, 4, 5, 6],                  # max depth
    'num_leaves':       [15, 20, 31, 50],               # main LightGBM complexity parameter
    'subsample':        [0.6, 0.7, 0.8, 1.0],          # row subsampling
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],          # feature subsampling
    'min_child_samples':[5, 10, 20],                   # min samples per leaf — regularisation
    'scale_pos_weight': [spw, 0.5, 0.637, 1.0],        # class weight variants
}

lgb_base = LGBMClassifier(random_state=42, verbose=-1)
lgb_search = RandomizedSearchCV(
    lgb_base, lgb_param_grid,
    n_iter=50, scoring='roc_auc',
    cv=cv, random_state=42, n_jobs=-1, verbose=1
)
lgb_search.fit(X_train_s, y_train)  # run the search

lgb_best = lgb_search.best_estimator_
y_prob_lgb_tuned = lgb_best.predict_proba(X_test_s)[:,1]
y_pred_lgb_tuned = lgb_best.predict(X_test_s)

print('LIGHTGBM TUNED')
print(f'  Best CV AUC:  {lgb_search.best_score_:.4f}')
print(f'  Test AUC:     {roc_auc_score(y_test, y_prob_lgb_tuned):.4f}')
print(f'  Test Acc:     {accuracy_score(y_test, y_pred_lgb_tuned)*100:.2f}%')
print(f'  Best params:  {lgb_search.best_params_}')
print()
print(classification_report(y_test, y_pred_lgb_tuned, target_names=['Low Injury','High Injury']))


Fitting 5 folds for each of 50 candidates, totalling 250 fits


/Users/askkarr/Desktop/final_year_project/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/askkarr/Desktop/final_year_project/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/askkarr/Desktop/final_year_project/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/askkarr/Desktop/final_year_project/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/askkarr/Desktop/final_year_project/.venv/lib/python3.9/site-packages/sklearn/utils/valida

LIGHTGBM TUNED
  Best CV AUC:  0.6619
  Test AUC:     0.6273
  Test Acc:     68.58%
  Best params:  {'subsample': 0.8, 'scale_pos_weight': 0.637, 'num_leaves': 50, 'n_estimators': 200, 'min_child_samples': 10, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 0.8}

              precision    recall  f1-score   support

  Low Injury       0.47      0.35      0.40        78
 High Injury       0.75      0.83      0.79       183

    accuracy                           0.69       261
   macro avg       0.61      0.59      0.59       261
weighted avg       0.66      0.69      0.67       261



- CV AUC=0.6619, Test AUC=0.6273 — CV-to-test gap is 0.0346, the largest in this cycle. This indicates significant overfitting to the training folds despite regularization.
- Performance regression compared to baseline (0.6355 → 0.6273) — unlike XGBoost, tuning actually decreased test performance here. The model became too specialized in the training patterns.
- Complexity vs. Data Size — The best parameters favored num_leaves=50. For a dataset of only 1,040 rows, this leaf-wise complexity likely "memorized" noise in the training set rather than finding general patterns.
- LightGBM is highly sensitive to small data. While it had the strongest untuned score, the tuning process struggled to find a configuration that improved on the robust default settings.

## Full Results Comparison — Tuned vs Baseline

Trains a Logistic Regression baseline (same config as the modelling notebook), then builds a summary table comparing all tuned models to the LR baseline.

In [9]:
# Train LR baseline as a reference point
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
y_prob_lr = lr.predict_proba(X_test_s)[:,1]     # LR injury probabilities
lr_auc = roc_auc_score(y_test, y_prob_lr)        # LR AUC

results = pd.DataFrame([
    {'Model': 'Logistic Regression (Baseline)', 'Type': 'Baseline', 'AUC-ROC': lr_auc,                                   'CV-Test Gap': 0.0000},
    {'Model': 'XGBoost Tuned',                  'Type': 'Tuned',    'AUC-ROC': roc_auc_score(y_test, y_prob_xgb_tuned), 'CV-Test Gap': xgb_search.best_score_ - roc_auc_score(y_test, y_prob_xgb_tuned)},
    {'Model': 'Random Forest Tuned',            'Type': 'Tuned',    'AUC-ROC': roc_auc_score(y_test, y_prob_rf_tuned),  'CV-Test Gap': rf_search.best_score_ - roc_auc_score(y_test, y_prob_rf_tuned)},
    {'Model': 'LightGBM Tuned',                 'Type': 'Tuned',    'AUC-ROC': roc_auc_score(y_test, y_prob_lgb_tuned), 'CV-Test Gap': lgb_search.best_score_ - roc_auc_score(y_test, y_prob_lgb_tuned)},
])
results['AUC-ROC'] = results['AUC-ROC'].round(4)
results['CV-Test Gap'] = results['CV-Test Gap'].round(4)  # positive gap = overfitting
print(results.sort_values('AUC-ROC', ascending=False).to_string(index=False))
print()
best_tuned = results[results['Type'] == 'Tuned'].loc[results['AUC-ROC'].idxmax()]  # best tuned model
print(f'=== BEST TUNED MODEL ===')
print(f'Model: {best_tuned["Model"]}')
print(f'AUC: {best_tuned["AUC-ROC"]}')
print(f'Improvement over LR baseline: +{(best_tuned["AUC-ROC"] - lr_auc):.4f}')


                         Model     Type  AUC-ROC  CV-Test Gap
                 XGBoost Tuned    Tuned   0.6558       0.0026
                LightGBM Tuned    Tuned   0.6273       0.0346
Logistic Regression (Baseline) Baseline   0.6220       0.0000
           Random Forest Tuned    Tuned   0.6169       0.0241

=== BEST TUNED MODEL ===
Model: XGBoost Tuned
AUC: 0.6558
Improvement over LR baseline: +0.0338


## Save Best Model

Identifies the highest-AUC tuned model and saves three artefacts: model, scaler, and feature column list.

In [10]:
# Find the best model by AUC
xgb_auc = roc_auc_score(y_test, y_prob_xgb_tuned)
rf_auc = roc_auc_score(y_test, y_prob_rf_tuned)
lgb_auc = roc_auc_score(y_test, y_prob_lgb_tuned)

best_auc = max(xgb_auc, rf_auc, lgb_auc)  # highest AUC wins
model_map = {
    xgb_auc: (xgb_best, 'XGBoost Tuned'),
    rf_auc: (rf_best, 'Random Forest Tuned'),
    lgb_auc: (lgb_best, 'LightGBM Tuned'),
}
best_model, best_name = model_map[best_auc]  # retrieve model and name

print(f'Best model: {best_name}')
print(f'AUC: {best_auc:.4f}')
print(f'Improvement over LR baseline: +{(best_auc - lr_auc):.4f}')
print()

model_dir = str(Paths.MODELS_C3)
os.makedirs(model_dir, exist_ok=True)  # create directory if missing

# joblib.dump(best_model, os.path.join(model_dir, 'cycle3_best_model.pkl'))    # save model
# joblib.dump(scaler, os.path.join(model_dir, 'cycle3_scaler.pkl'))             # save scaler
# joblib.dump(list(X_train.columns), os.path.join(model_dir, 'cycle3_feature_cols.pkl'))  # save feature list

# print('Saved:')
# print(f'  {model_dir}/cycle3_best_model.pkl')
# print(f'  {model_dir}/cycle3_scaler.pkl')
# print(f'  {model_dir}/cycle3_feature_cols.pkl')


Best model: XGBoost Tuned
AUC: 0.6558
Improvement over LR baseline: +0.0338



Best model: **XGBoost Tuned, AUC=0.6558** (within the expected 0.60–0.70 range for injury prediction)  
- I do not save the final model from this notebook. The final production model is saved from the Chronological Split notebook, as it provides a more realistic evaluation for football prediction.